# LangChain: Deep Technical Blog — Code Examples

**Author:** Imam  
**Internship:** Data Science Intern @ Innomatics Research Labs  
**Blog:** [Read the full blog on Medium](https://medium.com/@imamsab.im22/langchain-ffcb16610c92)

---

This notebook contains all working code examples from the blog post.  
Each section maps directly to a component explained in the blog.

### Sections
1. Installation & Setup
2. Basic LLM Call
3. Prompt Templates
4. Chains (LCEL)
5. Memory
6. Document Loaders & Vector Stores (RAG)
7. Agents & Tools

---
## 1. Installation & Setup

In [ ]:
# Install all required dependencies
!pip install langchain langchain-openai langchain-community faiss-cpu openai -q

In [ ]:
import os

# Set your OpenAI API key here
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"

print("Setup complete!")

---
## 2. Basic LLM Call

The simplest possible interaction with an LLM using LangChain.  
We use `ChatOpenAI` which accepts structured messages (System + Human) instead of raw strings.

> **Why this matters:** LangChain wraps the raw API call into a clean interface.  
> Swapping to a different model (HuggingFace, Anthropic, etc.) only requires changing one line.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# Initialize the chat model
# temperature=0.7 → balanced between creative and deterministic
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7
)

# Build the message list
messages = [
    SystemMessage(content="You are a concise technical assistant. Keep answers under 3 sentences."),
    HumanMessage(content="What is a transformer in deep learning?")
]

# Invoke the model
response = llm.invoke(messages)

print("Response:")
print(response.content)
print(f"\nModel used: {response.response_metadata.get('model_name', 'N/A')}")

---
## 3. Prompt Templates

Prompt Templates let you define reusable prompt structures with dynamic variables.  
Instead of hardcoding strings, you define a template once and fill it with different values at runtime.

> **Why this matters:** In real apps, you have dozens of prompt patterns.  
> Templates keep them organized, testable, and consistent.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Define a reusable template with {variables}
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. Answer concisely and accurately."),
    ("human", "Explain the concept of {concept} in simple terms with one real-world analogy.")
])

# Example 1: Machine Learning domain
formatted_1 = prompt.format_messages(
    domain="machine learning",
    concept="gradient descent"
)

response_1 = llm.invoke(formatted_1)
print("Example 1 — Gradient Descent:")
print(response_1.content)
print()

# Example 2: Same template, different domain — no code rewrite needed
formatted_2 = prompt.format_messages(
    domain="databases",
    concept="indexing"
)

response_2 = llm.invoke(formatted_2)
print("Example 2 — Database Indexing:")
print(response_2.content)

---
## 4. Chains (LCEL — LangChain Expression Language)

Chains connect components so the output of one flows directly into the next.  
LCEL uses the pipe `|` operator to wire everything together cleanly.

**Flow:** `PromptTemplate → LLM → OutputParser`

> **Why this matters:** What used to take 30+ lines of plumbing is now 3 lines.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

# Define the prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that summarizes text clearly."),
    ("human", "Summarize the following in exactly one sentence:\n\n{text}")
])

# Build the chain using LCEL pipe syntax
# prompt → llm → parse output as plain string
chain = prompt | llm | StrOutputParser()

# Run the chain
result = chain.invoke({
    "text": """
    LangChain is an open-source framework that helps developers build applications 
    powered by large language models. It provides abstractions for prompting, 
    memory management, chaining multiple LLM calls, connecting to external tools 
    and databases, and orchestrating complex AI workflows — all through a modular, 
    composable API.
    """
})

print("Summary:")
print(result)

In [ ]:
# Multi-step chain: Translate → Summarize
# Demonstrates how chains can be nested and reused

translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a translator. Translate the given text to English accurately."),
    ("human", "Translate this to English: {text}")
])

summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a summarizer."),
    ("human", "Summarize this in one sentence: {translated_text}")
])

# Chain 1: translate
translate_chain = translate_prompt | llm | StrOutputParser()

# Chain 2: pass translation into summarize
full_chain = translate_chain | (lambda x: {"translated_text": x}) | summarize_prompt | llm | StrOutputParser()

output = full_chain.invoke({"text": "La inteligencia artificial está transformando el mundo."})
print("Multi-step Chain Output:")
print(output)

---
## 5. Memory

LLMs are stateless by default — they forget everything between API calls.  
Memory fixes this by maintaining conversation history and injecting it into each new prompt.

> **Why this matters:** Without memory, a chatbot can't hold a basic multi-turn conversation.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# Manually managed conversation history
# Simple, transparent, and gives full control
chat_history = []

def chat(user_input: str) -> str:
    """Send a message and get a response, maintaining full conversation history."""
    
    # Build prompt with history injected via MessagesPlaceholder
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a friendly and helpful assistant. Use conversation history to stay consistent."),
        MessagesPlaceholder(variable_name="history"),  # ← history injected here
        ("human", "{input}")
    ])
    
    chain = prompt | llm
    
    response = chain.invoke({
        "history": chat_history,
        "input": user_input
    })
    
    # Update history with this exchange
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response.content))
    
    return response.content


# Test: multi-turn conversation
print("Turn 1:")
print(chat("Hi! My name is Imam and I'm learning LangChain."))
print()

print("Turn 2:")
print(chat("What topic am I currently studying?"))
print()

print("Turn 3:")
print(chat("What's my name again?"))

---
## 6. Document Loaders & Vector Stores (RAG)

RAG = Retrieval-Augmented Generation.  
It lets the LLM answer questions about your own documents — even thousands of pages worth.

**Flow:** `Load → Split → Embed → Store → Retrieve → Answer`

> **Why this matters:** You can't paste a 200-page PDF into a prompt.  
> RAG retrieves only the relevant chunks and feeds those to the model.

In [ ]:
import os

# Create a sample document to query against
sample_text = """
Company Leave Policy

Annual Leave:
All full-time employees are entitled to 20 days of paid annual leave per year.
Leave must be approved by the direct manager at least 5 working days in advance.
Unused leave can be carried forward for a maximum of 10 days to the next calendar year.

Sick Leave:
Employees are entitled to 10 days of paid sick leave per year.
A medical certificate is required for sick leave exceeding 3 consecutive days.
Sick leave cannot be carried forward to the next year.

Maternity and Paternity Leave:
Female employees are entitled to 26 weeks of paid maternity leave.
Male employees are entitled to 2 weeks of paid paternity leave.
These benefits apply after completing 6 months of continuous employment.

Public Holidays:
Employees are entitled to all gazetted public holidays.
If required to work on a public holiday, employees receive double pay.
"""

# Write to a temp file (simulates loading a real document)
with open("/tmp/company_policy.txt", "w") as f:
    f.write(sample_text)

print("Sample document created!")

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA

# ── Step 1: Load the document ─────────────────────────────────────────────────
loader = TextLoader("/tmp/company_policy.txt")
documents = loader.load()
print(f"Loaded {len(documents)} document(s)")

# ── Step 2: Split into chunks ─────────────────────────────────────────────────
# chunk_size: characters per chunk
# chunk_overlap: overlap between chunks to preserve context at boundaries
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)
chunks = splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")

# ── Step 3 & 4: Embed and store in FAISS (local vector store) ─────────────────
# OpenAIEmbeddings converts text → dense vectors
# FAISS stores them for fast similarity search
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)
print("Vector store created!")

# ── Step 5: Build the RetrievalQA chain ──────────────────────────────────────
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(
        search_kwargs={"k": 3}  # retrieve top 3 most relevant chunks
    )
)

print("\nRAG pipeline ready!")

In [ ]:
# Ask questions — the model retrieves relevant chunks before answering

questions = [
    "How many days of annual leave do employees get?",
    "What is the maternity leave policy?",
    "Can unused sick leave be carried forward?"
]

for q in questions:
    print(f"Q: {q}")
    answer = qa_chain.invoke({"query": q})
    print(f"A: {answer['result']}")
    print("-" * 60)

---
## 7. Agents & Tools

Agents are the most powerful part of LangChain.  
Instead of a fixed pipeline, an agent dynamically decides which tool to call based on the user's input.

**Flow:** `User Input → Agent reasons → Picks tool → Calls tool → Observes result → Answers`

> **Why this matters:** Agents can solve problems that require multiple steps and external data — things a static chain can't handle.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
import math

# ── Define Custom Tools ───────────────────────────────────────────────────────
# The @tool decorator converts a Python function into a LangChain tool
# The docstring is what the agent reads to decide when to use the tool

@tool
def calculator(expression: str) -> str:
    """
    Evaluates a mathematical expression and returns the result.
    Use this for any arithmetic or math calculation.
    Input should be a valid Python math expression, e.g. '2 ** 10' or '(15 * 3) + 7'.
    """
    try:
        # Safe evaluation — only allows math operations
        allowed = {"__builtins__": {}, "math": math}
        result = eval(expression, allowed)
        return f"Result: {result}"
    except Exception as e:
        return f"Calculation error: {str(e)}"


@tool
def word_counter(text: str) -> str:
    """
    Counts the number of words in a given text.
    Use this when the user asks how many words are in a sentence or paragraph.
    """
    words = text.strip().split()
    return f"The text contains {len(words)} words."


@tool
def string_reverser(text: str) -> str:
    """
    Reverses a given string or sentence.
    Use this when the user asks to reverse a word or phrase.
    """
    return f"Reversed: {text[::-1]}"


# ── Set up the Agent ──────────────────────────────────────────────────────────
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # temperature=0 for reliable tool use

tools = [calculator, word_counter, string_reverser]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant with access to tools. Always use the appropriate tool to answer accurately."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")  # ← agent's internal reasoning goes here
])

# Create the agent
agent = create_openai_tools_agent(llm, tools, prompt)

# AgentExecutor runs the agent in a loop until it reaches a final answer
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True  # prints the agent's reasoning steps
)

print("Agent ready!")

In [ ]:
# Test 1: Math calculation
print("=" * 60)
print("Query 1: Math")
print("=" * 60)

result = agent_executor.invoke({
    "input": "What is 1847 multiplied by 293?"
})
print(f"\nFinal Answer: {result['output']}")

In [ ]:
# Test 2: Multi-tool query — agent decides to use two different tools
print("=" * 60)
print("Query 2: Multi-tool")
print("=" * 60)

result = agent_executor.invoke({
    "input": "How many words are in 'the quick brown fox jumps over the lazy dog'? Also, what is 2 raised to the power 8?"
})
print(f"\nFinal Answer: {result['output']}")

In [ ]:
# Test 3: String reversal
print("=" * 60)
print("Query 3: String tool")
print("=" * 60)

result = agent_executor.invoke({
    "input": "Can you reverse the word 'LangChain'?"
})
print(f"\nFinal Answer: {result['output']}")

---
## Summary

| Component | What it does | Key Class |
|---|---|---|
| LLM / Chat Model | The brain — generates text | `ChatOpenAI` |
| Prompt Template | Reusable, dynamic prompts | `ChatPromptTemplate` |
| Chain (LCEL) | Connects components with `\|` | `prompt \| llm \| parser` |
| Memory | Persists conversation history | `MessagesPlaceholder` |
| Document Loader | Loads files into LangChain | `TextLoader`, `PDFLoader` |
| Vector Store | Stores and retrieves embeddings | `FAISS`, `Pinecone` |
| Agent | Dynamically picks and calls tools | `AgentExecutor` |
| Tool | A function the agent can call | `@tool` decorator |

---

**Read the full blog:** https://medium.com/@imamsab.im22/langchain-ffcb16610c92  
**Author:** Imam | Data Science Intern @ Innomatics Research Labs | February 2026